In [90]:
from glob import glob
import cmocean.cm as cmo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from scipy.fft import fft, fftfreq

import matplotlib.style as mplstyle
mplstyle.use(["ggplot", "fast"])

In [91]:
water_height_folders = ['20 cm', '20 cm no plates', '30 cm', '30 cm no plates', '40 cm', '40 cm no plates']
water_heights = np.asarray([0.2, 0.3, 0.4])  #[m]

In [92]:
# Defining some constants for later
H = 0.05  # height of plates [m]
D_H = water_heights - H  # depth of porous layer

g = 9.81  # Gravity [m^2/s]
h = 0.00122  # Characteristic height [m]
nu = 10e-6  # Kinematic viscosity of water at 20 deg C [m^2/s]

In [93]:
# Choose a water depth to work with
water_height = water_height_folders[5]
w_h = water_heights[-1]

In [94]:
# Path to files
basepath = glob('/Users/kjesta/Desktop/Master prosjekt/Maxime_sine_greier/Maxime-s-Programs')[0]
positions = ['P0', 'P1', 'P2', 'P3', 'P4']
runs = ['run1', 'run2', 'run3']

if water_height == water_height_folders[0] or water_height == water_height_folders[1]:
    # Frequencies for the given depth
    frequencies = ['f076', 'f092', 'f107', 'f122', 'f137', 'f153']
    amplitude = 'A015'

if water_height == water_height_folders[2] or water_height == water_height_folders[3]:
    # Frequencies for the given depth
    frequencies = ['f059', 'f071', 'f083', 'f095', 'f106', 'f118']
    amplitude = 'A03' 

if water_height == water_height_folders[4]:
    # Frequencies for the given depth
    frequencies = ['f1', 'f06', 'f06', 'f07', 'f08', 'f09']
    amplitude = 'A03'

if water_height == water_height_folders[5]:
    # Frequencies for the given depth
    frequencies = ['f1', 'f05', 'f06', 'f07', 'f08', 'f09']
    amplitude = 'A030'

In [95]:
# Choose a frequency to work with:
freq = frequencies[1]
f = 0.6  # [Hz] = [s^-1]

# Choose a rack position to work with:
pos = positions[0]

In [96]:
filepaths = []
for run in runs:
    filepath = f'{basepath}/{water_height}/' + pos + f'/{freq}/{freq}_{amplitude}_{pos}_{run}.csv'
    filepaths.append(filepath)

In [97]:
headers = ['time', 'p1', 'p2', 'p3', 'p4', 'speed of sound']

df_run1 = pd.read_csv(filepaths[0], header=None)
df_run2 = pd.read_csv(filepaths[1], header=None)
df_run3 = pd.read_csv(filepaths[2], header=None)

df_run1.columns = headers
df_run2.columns = headers
df_run3.columns = headers

FileNotFoundError: [Errno 2] No such file or directory: '/Users/kjesta/Desktop/Master prosjekt/Maxime_sine_greier/Maxime-s-Programs/40 cm no plates/P0/f05/f05_A030_P0_run1.csv'

In [ ]:
def modify(df):
    '''
    Removes noise from data by subtracting the mean of the first 200 rows,
    and converts timestamps to datetime values.
    '''
    # Removing noise
    for col in range(1, 5):
        noise = df.iloc[:200, col].mean()
        df.iloc[:, col] = df.iloc[:, col] - noise

    # Fixing the timestamps
    df['time'] = pd.to_datetime(df['time'])

    return df

In [ ]:
run1 = modify(df_run1)
run2 = modify(df_run2)
run3 = modify(df_run3)

In [ ]:
# Applying low pass filter in the same style as Maxime
weights = np.array([0.1, 0.2, 0.4, 0.2, 0.1])

def processing(df, tol=1e-6, max_gap=4, passes=2):
    ''' 
    Corrects for large outliers and smoothes the data
    '''
    df_smooth = df.copy()

    for col in df_smooth.columns[1:5]:
        values = df_smooth[col].values.astype(float).copy()

        n = len(values)
        for gap in range(1, max_gap + 1):
            for i in range(1, n - gap - 1):
                left = values[i - 1]
                right = values[i + gap]
                if abs(left - right) < tol:
                    middle = values[i:i + gap]
                    if all(abs(m - left) > tol for m in middle):
                        values[i:i + gap] = left

        for _ in range(passes):
            smoothed = values.copy()
            for i in range(2, len(values) - 2):
                window = values[i-2:i+3]
                smoothed[i] = np.dot(weights, window)
            values = smoothed

        df_smooth[col] = values
    
    return df_smooth

In [ ]:
# Processed data
run1_p = processing(run1)
run2_p = processing(run2)
run3_p = processing(run3)

In [ ]:
X_base = np.array([11.075, 11.38, 11.685, 11.99])  # Base positions [m]
offset = 1.215

if pos == positions[0]:
    probe_positions = X_base

if pos == positions[1]:
    probe_positions = X_base + offset

if pos == positions[2]:
    probe_positions = X_base + offset*2

if pos == positions[3]:
    probe_positions = X_base + offset*3

if pos == positions[4]:
    probe_positions = X_base + offset*4

beach_start = 12.4
beach_end = 15.45

In [ ]:
def estimate_wavelength(df, probe_positions):
    signals = [[],[],[],[]]  # for storing probe signals

    for i in range(1, 5):
        signals[i-1] = df.iloc[:, i].values
    
    signals = np.array(signals)
    n_probes = 4

    total_time = (df['time'].iloc[-1] - df['time'].iloc[0]).total_seconds()
    dt = total_time / len(df['time'])  # time step
    fs = 1 / dt  # sample frequency
    n = len(df['time'])
    
    # FFT of each signal
    fft_signals = fft(signals, axis=1)
    freqs = fftfreq(n, d=dt)
    
    # Keep only positive frequencies
    pos_freqs = freqs[:n//2]
    fft_pos = fft_signals[:, :n//2]

    
    # Find the dominant frequency (for example, on the signal of the 1st probe)
    power = np.abs(fft_pos[0])**2
    dominant_idx = np.argmax(power[1:]) + 1  # ignore DC (frequency 0)
    f_dom = pos_freqs[dominant_idx]
    omega_dom = 2 * np.pi * f_dom

    # Extract phases at this frequency for each probe
    phases = np.angle(fft_pos[:, dominant_idx])
    unwrapped_phases = np.unwrap(phases)

    # Linear fit: phase = k * x + phi0 → slope = k
    coeffs = np.polyfit(probe_positions, unwrapped_phases, 1)
    
    k = coeffs[0]  # wave number (rad/m)

    # Wavelength
    wavelength = 2 * np.pi / np.abs(k)
    
    return wavelength, f_dom

In [ ]:
def solve_disp_for_H(f, l):
    g = 9.81
    k = (2*np.pi)/l
    omega = 2*np.pi*f
    R = (omega**2)/(g*k)
    H_eff = np.arctanh(R)/k

    return H_eff

In [ ]:
l_1, f_1 = estimate_wavelength(run1_p, probe_positions)
l_2, f_2 = estimate_wavelength(run2_p, probe_positions)
l_3, f_3 = estimate_wavelength(run3_p, probe_positions)

In [ ]:
H_eff_1 = solve_disp_for_H(f_1, l_1)
print(f'Effective depth: {H_eff_1}')

H_eff_2 = solve_disp_for_H(f_2, l_2)
print(f'Effective depth: {H_eff_2}')

H_eff_3 = solve_disp_for_H(f_3, l_3)
print(f'Effective depth: {H_eff_3}')

Effective depth: 0.39520771080066425
Effective depth: 0.3949347194785756
Effective depth: 0.39456729875577223
